Türkçe ile İngilizceyi karşılaştır. Aynı paragrafın Türkçesini ve İngilizcesini al. GPT-4'ün tokenizer'ıyla (tiktoken, "cl100k_base") ikisini de token'la, kelime başına kaç token düştüğünü yaz. Sonra aynı Türkçe paragrafı kendi tokenizer'ınla token'la. Türkçenin eklemeli yapısı bu sayıları nasıl etkiliyor?

In [1]:
import tiktoken
import regex as re
import pandas as pd

In [2]:
tr = ("Geçen yaz ailemle birlikte Karadeniz'e gittik. Yeşil yaylalarda saatlerce yürüdük, "
      "sabahları sisli dağların arasında demlenmiş sıcak çay içtik. Akşamları köydeki yaşlılar "
      "bize çocukluklarından kalan eski hikâyeleri anlattılar. Şehre döndüğümüzde hepimiz o "
      "sessizliği ve temiz havayı ne kadar özlediğimizi fark ettik.")

en = ("Last summer we went to the Black Sea with my family. We walked for hours in the green "
      "highlands, and in the mornings we drank hot brewed tea among the misty mountains. In the "
      "evenings, the elderly people in the village told us old stories left from their childhood. "
      "When we returned to the city, we all realized how much we had missed that silence and the clean air.")

print(tr, "\n")
print(en)

Geçen yaz ailemle birlikte Karadeniz'e gittik. Yeşil yaylalarda saatlerce yürüdük, sabahları sisli dağların arasında demlenmiş sıcak çay içtik. Akşamları köydeki yaşlılar bize çocukluklarından kalan eski hikâyeleri anlattılar. Şehre döndüğümüzde hepimiz o sessizliği ve temiz havayı ne kadar özlediğimizi fark ettik. 

Last summer we went to the Black Sea with my family. We walked for hours in the green highlands, and in the mornings we drank hot brewed tea among the misty mountains. In the evenings, the elderly people in the village told us old stories left from their childhood. When we returned to the city, we all realized how much we had missed that silence and the clean air.


### GPT-4 tokenizer (cl100k_base)

In [3]:
enc = tiktoken.get_encoding("cl100k_base")

def ozet(ad, metin, ids):
    kelime = len(metin.split())
    return {"metin": ad, "kelime": kelime, "karakter": len(metin),
            "byte": len(metin.encode("utf-8")), "token": len(ids),
            "token/kelime": round(len(ids) / kelime, 2),
            "byte/token": round(len(metin.encode("utf-8")) / len(ids), 2)}

tr_gpt4 = enc.encode(tr)
en_gpt4 = enc.encode(en)
pd.DataFrame([ozet("TR - cl100k_base", tr, tr_gpt4), ozet("EN - cl100k_base", en, en_gpt4)])

,metin,kelime,karakter,byte,token,token/kelime,byte/token
0,TR - cl100k_base,40,316,348,135,3.38,2.58
1,EN - cl100k_base,69,366,366,78,1.13,4.69


In [4]:
# token'ların hangi parçalara karşılık geldiği
print("TR:", [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in tr_gpt4])
print()
print("EN:", [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in en_gpt4])

TR: ['Ge', 'ç', 'en', ' yaz', ' a', 'ile', 'm', 'le', ' bir', 'lik', 'te', ' Kar', 'aden', 'iz', "'e", ' g', 'itt', 'ik', '.', ' Ye', 'ş', 'il', ' yay', 'l', 'al', 'arda', ' saat', 'ler', 'ce', ' y', 'ür', 'ü', 'd', 'ük', ',', ' sab', 'ah', 'ları', ' sis', 'li', ' da', 'ğ', 'ların', ' ar', 'as', 'ında', ' dem', 'len', 'mi', 'ş', ' s', 'ıc', 'ak', ' ç', 'ay', ' iç', 't', 'ik', '.', ' Ak', 'ş', 'aml', 'arı', ' kö', 'y', 'de', 'ki', ' yaş', 'lı', 'lar', ' b', 'ize', ' ç', 'oc', 'uk', 'l', 'uk', 'ları', 'nd', 'an', ' k', 'alan', ' es', 'ki', ' h', 'ik', 'â', 'y', 'eler', 'i', ' an', 'l', 'att', 'ı', 'lar', '.', ' Ş', 'eh', 're', ' d', 'ö', 'nd', 'ü', 'ğ', 'üm', 'üz', 'de', ' hep', 'im', 'iz', ' o', ' sess', 'iz', 'li', 'ği', ' ve', ' tem', 'iz', ' hav', 'ay', 'ı', ' ne', ' kadar', ' ö', 'z', 'led', 'i', 'ğ', 'im', 'izi', ' f', 'ark', ' ett', 'ik', '.']

EN: ['Last', ' summer', ' we', ' went', ' to', ' the', ' Black', ' Sea', ' with', ' my', ' family', '.', ' We', ' walked', ' for', ' hours

### Kendi tokenizer'ım (GPT-4 regex + BPE, Felatun Bey metninde eğitildi, vocab 1000)

In [5]:
with open('felatun_bey_tam_metin.txt', 'r', encoding='utf-8') as f:
    text = f.read()

GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
gpt4pat = re.compile(GPT4_SPLIT_PATTERN)

def get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [6]:
def merge(ids, pair, idx):
    # ids içinde art arda gelen pair'leri yeni idx token'ı ile değiştir
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [7]:
vocab_size = 1000
num_merges = vocab_size - 256

ids_chunks = [list(ch.encode("utf-8")) for ch in gpt4pat.findall(text)]
merges = {}
for i in range(num_merges):
    stats = {}
    for chunk_ids in ids_chunks:
        get_stats(chunk_ids, stats)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids_chunks = [merge(chunk_ids, pair, idx) for chunk_ids in ids_chunks]
    merges[pair] = idx

vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

In [8]:
def encode_chunk(text_bytes):
    ids = list(text_bytes)
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids

def encode(text):
    ids = []
    for chunk in gpt4pat.findall(text):
        ids.extend(encode_chunk(chunk.encode("utf-8")))
    return ids

def decode(ids):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

tr_mine = encode(tr)
en_mine = encode(en)
print(decode(tr_mine) == tr)
print("TR:", [vocab[i].decode("utf-8", errors="replace") for i in tr_mine])

True
TR: ['G', 'e', 'ç', 'en', ' yaz', ' a', 'ile', 'm', 'le', ' bir', 'lik', 'te', ' K', 'ar', 'ad', 'eniz', "'", 'e', ' git', 't', 'ik', '.', ' Y', 'eş', 'il', ' y', 'ay', 'l', 'alar', 'da', ' saat', 'ler', 'ce', ' y', 'ür', 'ü', 'd', 'ük', ',', ' s', 'ab', 'ah', 'ları', ' s', 'is', 'li', ' da', 'ğ', 'ların', ' ar', 'as', 'ında', ' de', 'm', 'len', 'miş', ' s', 'ı', 'cak', ' ç', 'ay', ' iç', 't', 'ik', '.', ' A', 'k', 'şam', 'ları', ' k', 'ö', 'y', 'de', 'ki', ' yaş', 'lı', 'lar', ' biz', 'e', ' çocuk', 'luk', 'lar', 'ından', ' kal', 'an', ' es', 'ki', ' h', 'ik', 'ây', 'eler', 'i', ' anla', 't', 'tı', 'lar', '.', ' Ş', 'eh', 'r', 'e', ' dö', 'n', 'dü', 'ğ', 'üm', 'üz', 'de', ' h', 'ep', 'imiz', ' o', ' s', 'ess', 'iz', 'li', 'ği', ' ve', ' t', 'em', 'iz', ' ha', 'v', 'ayı', ' ne', ' kadar', ' öz', 'le', 'diğ', 'imiz', 'i', ' f', 'ar', 'k', ' ett', 'ik', '.']


### Karşılaştırma tablosu

In [9]:
pd.DataFrame([
    ozet("TR - cl100k_base (100k vocab)", tr, tr_gpt4),
    ozet("EN - cl100k_base (100k vocab)", en, en_gpt4),
    ozet("TR - benim (1000 vocab)", tr, tr_mine),
    ozet("EN - benim (1000 vocab)", en, en_mine),
])

,metin,kelime,karakter,byte,token,token/kelime,byte/token
0,TR - cl100k_base (100k vocab),40,316,348,135,3.38,2.58
1,EN - cl100k_base (100k vocab),69,366,366,78,1.13,4.69
2,TR - benim (1000 vocab),40,316,348,137,3.42,2.54
3,EN - benim (1000 vocab),69,366,366,238,3.45,1.54


### Kelime kelime: eklemeli yapı

In [10]:
def parcalar_gpt4(w):
    return [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in enc.encode(w)]

def parcalar_benim(w):
    return [vocab[i].decode("utf-8", errors="replace") for i in encode(w)]

ekler = [" ev", " evler", " evlerimiz", " evlerimizden", " evlerimizdenmiş",
         " göz", " gözlük", " gözlükçü", " gözlükçülük", " gözlükçülükten",
         " özlediğimizi", " çocukluklarından", " Çekoslovakyalılaştıramadıklarımızdanmışsınız"]
rows = []
for w in ekler:
    rows.append({"kelime": w.strip(), "cl100k token": len(enc.encode(w)), "cl100k parçalar": parcalar_gpt4(w),
                 "benim token": len(encode(w)), "benim parçalar": parcalar_benim(w)})
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(rows)

,kelime,cl100k token,cl100k parçalar,benim token,benim parçalar
0,ev,1,[ ev],1,[ ev]
1,evler,2,"[ ev, ler]",2,"[ ev, ler]"
2,evlerimiz,4,"[ ev, ler, im, iz]",3,"[ ev, ler, imiz]"
3,evlerimizden,5,"[ ev, ler, im, iz, den]",4,"[ ev, ler, imiz, den]"
4,evlerimizdenmiş,7,"[ ev, ler, im, iz, den, mi, ş]",5,"[ ev, ler, imiz, den, miş]"
5,göz,2,"[ gö, z]",1,[ göz]
6,gözlük,3,"[ gö, zl, ük]",3,"[ göz, l, ük]"
7,gözlükçü,5,"[ gö, zl, ük, ç, ü]",5,"[ göz, l, ük, ç, ü]"
8,gözlükçülük,6,"[ gö, zl, ük, ç, ül, ük]",6,"[ göz, l, ük, ç, ül, ük]"
9,gözlükçülükten,7,"[ gö, zl, ük, ç, ül, ük, ten]",7,"[ göz, l, ük, ç, ül, ük, ten]"


In [11]:
# aynı anlam: Türkçede tek kelime, İngilizcede birçok kelime
for t, e in [(" evlerimizden", " from our houses"),
             (" gidebileceklerimizden", " from those we will be able to go"),
             (" özlediğimizi", " that we missed")]:
    print(f"{t.strip():25} cl100k: {len(enc.encode(t))} token  |  {e.strip():35} cl100k: {len(enc.encode(e))} token")

evlerimizden              cl100k: 5 token  |  from our houses                     cl100k: 3 token
gidebileceklerimizden     cl100k: 9 token  |  from those we will be able to go    cl100k: 8 token
özlediğimizi              cl100k: 7 token  |  that we missed                      cl100k: 3 token


### Yorum

**cl100k_base (GPT-4) sonuçları**
| | kelime | token | token/kelime |
|---|---|---|---|
| Türkçe | 40 | 135 | **3.38** |
| İngilizce | 69 | 78 | **1.13** |

Aynı paragraf Türkçede %42 daha az kelimeyle yazılıyor ama **1.73 kat daha fazla token** tutuyor. İngilizcede neredeyse her kelime tek token (` summer`, ` mornings`, ` childhood`, ` realized`), Türkçede ise kelimeler parçalanıyor: ` g | itt | ik`, ` ö | z | led | i | ğ | im | izi`, ` ç | oc | uk | l | uk | ları | nd | an`. Byte başına da fark büyük: İngilizcede bir token ortalama 4.69 byte, Türkçede 2.58 byte taşıyor.

**Kendi tokenizer'ım (GPT-4 regex + BPE, vocab 1000, sadece Felâtun Bey ile eğitildi)**
Türkçe paragraf **137 token, 3.42 token/kelime**, yani 100 kat büyük vocab'a sahip GPT-4 ile neredeyse aynı (135). Bunun sebebi cl100k_base'in vocab'ının büyük kısmının İngilizce ve kod için harcanmış olması; benim 744 birleştirmemin hepsi Türkçe. Üstelik benim parçalarım dilbilgisel olarak daha anlamlı: ` çocuk | luk | lar | ından`, ` ev | ler | imiz | den | miş`, ` öz | le | diğ | imiz | i`. GPT-4 ise ` çocuklarından`'ı 8 anlamsız parçaya bölüyor. Tersine, İngilizce paragraf benim tokenizer'ımla 238 token'a (3.45 token/kelime) çıkıyor; tokenizer hangi dilde eğitildiyse o dili sıkıştırıyor.

**Eklemeli yapının etkisi**
- Türkçede bir kelime = kök + art arda ekler. Aynı kökten çok sayıda farklı kelime biçimi çıkıyor (`ev`, `evler`, `evlerimiz`, `evlerimizden`, `evlerimizdenmiş` ...). Her biçim tek başına nadir olduğu için BPE bunları vocab'a tam kelime olarak alamıyor, sadece sık ekleri (`ler`, `imiz`, `den`, `miş`) öğreniyor. Bu yüzden ek sayısı arttıkça token sayısı da neredeyse doğrusal artıyor: `ev` 1 → `evlerimizdenmiş` 5 (benim) / 7 (GPT-4) token. `Çekoslovakyalılaştıramadıklarımızdanmışsınız` tek kelime ama 21-24 token.
- İngilizce aynı anlamı ayrı ve çok sık geçen kelimelerle ifade ediyor (`from our houses` = 3 token, `evlerimizden` = 5 token; `that we missed` = 3, `özlediğimizi` = 7). İngilizce kelimeler sık olduğu için her biri vocab'da tek token olarak bulunuyor.
- Bu yüzden token/kelime oranı Türkçede doğal olarak yüksek çıkıyor; daha adil ölçü aynı içerik için toplam token sayısı (135'e 78). Türkçe harflerin 2 byte olması ve vocab'da yoksa ayrı token'lara düşmesi (` da | ğ | ların`) de sayıyı artırıyor.
- Sonuç olarak aynı içerik için Türkçe metin context penceresini daha hızlı dolduruyor, API maliyeti daha yüksek oluyor ve üretim daha yavaş. Türkçe ağırlıklı ve daha büyük vocab'la eğitilmiş bir tokenizer bu farkı büyük ölçüde kapatabilir; 1000'lik vocab'ım bile GPT-4 seviyesine geldi.